# 2 — Architecture notebook (template)

Copy this file into `notebooks/architectures/<your-direction>/` and rename it
`train_<direction>.ipynb`. Then work only in your copy.

**Rules** (from `lie-detector/AGENTS.md`)

1. Don't touch `data/processed/splits/` or `lib/` — if you need a change there, tell the group.
2. Your model lives here, or in a `model.py` beside this notebook if it gets long.
3. **Results come from the five folds.** This notebook never touches the test set.
4. Save predictions per clip: they are what makes the directions comparable.

What you replace: cells marked `YOUR CODE`. Everything else stays as it is.

## 0 — Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# The repo is needed for two things that are NOT on Drive: the `lib/` code and the
# frozen splits (they are versioned in git, that is what makes them frozen).
import os
if not os.path.isdir("/content/DeepLearningCourse/.git"):
    !git clone -q https://github.com/mattsays/DeepLearningCourse /content/DeepLearningCourse
%cd /content/DeepLearningCourse/lie-detector
!git pull -q
import sys; sys.path.insert(0, '.')
#%load_ext autoreload
#%autoreload 2

/content/DeepLearningCourse/lie-detector


In [4]:
from pathlib import Path

# The one shared folder. Same path for everybody: add a Drive shortcut named
# "dolos" in your My Drive and this resolves without touching anything else.
DATA = Path("/content/drive/MyDrive/dolos")
print(DATA, "->", sorted(p.name for p in DATA.glob("*")))

/content/drive/MyDrive/dolos -> ['audio', 'clips', 'dolos_annotations.zip', 'splits']


In [ ]:
from lib.dataset import cache_dir

# YOUR CODE: rename these
DIRECTION = "my-direction"
ENCODER = "wav2vec2-base"           # the model that encodes the features
CACHE = cache_dir(ENCODER, DATA)    # DATA/cached_features/<ENCODER>/<clip_id>.npy
SEED = 0
BATCH_SIZE = 16
EPOCHS = 20
PATIENCE = 4
FOLDS = 5
print(CACHE)

## 1 — The frozen splits

`Protocol.load(DATA / "splits")` reads the protocol from the shared folder and verifies
the manifest hash: a file edited by hand raises instead of silently giving you a different split.

In [ ]:
from lib.dataset import Protocol, Clips

folds = Protocol.load(DATA / "splits")
clips = Clips.load(DATA)
print(clips.describe_source())   # post it: same fingerprint = same dataset

train, val, holdout = folds.subject(1)
split_ids = [c for fold in range(1, FOLDS + 1)
             for part in folds.subject(fold) for c in part] + folds.test()
missing = clips.missing_from(split_ids)
print(f"{len(missing)} split clips are absent from dataset.csv")
if missing:
    print("  first:", missing[:3], "-> the splits and the data do not match, stop here")

for name, ids in (("train", train), ("val", val), ("holdout", holdout)):
    print(f"{name:8}", clips.describe(ids))

## 2 — Your features

The loader hands out whatever your preprocessing saved as `<clip_id>.npy`; it does
not care what is inside. If the cache does not exist yet, the notebook falls back to
deterministic pseudo-features so you can test the loop end to end — **the numbers are
meaningless in that mode**, it only proves the plumbing works.

In [ ]:
import hashlib
import numpy as np
import torch
from torch.utils.data import Dataset
from lib.dataset import CachedFeatures, make_loader

DEBUG = not CACHE.exists()
if DEBUG:
    print("⚠️  no cache at", CACHE, "-> pseudo-features, numbers are meaningless")


class PseudoFeatures(Dataset):
    # deterministic stand-in keyed by clip id, so the loop is reproducible

    def __init__(self, ids, clips, dim=32):
        self.ids = clips.keep(ids)
        self.clips = clips
        self.dim = dim

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        clip_id = self.ids[index]
        seed = int(hashlib.md5(clip_id.encode()).hexdigest()[:8], 16)
        rng = np.random.default_rng(seed)
        x = torch.tensor(rng.normal(size=self.dim), dtype=torch.float32)
        return x, torch.tensor(self.clips[clip_id].y, dtype=torch.long)


def loader_for(ids, shuffle):
    dataset = PseudoFeatures(ids, clips) if DEBUG else CachedFeatures(ids, clips, CACHE)
    return make_loader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, seed=SEED)

## 3 — The model

`YOUR CODE`. Keep the interface: `forward(x)` returns one logit per clip, shape `(batch,)`.
The placeholder below is deliberately trivial — it is not a suggestion.

In [ ]:
import torch.nn as nn


class Model(nn.Module):
    def __init__(self, feature_dim):
        super().__init__()
        self.head = nn.Linear(feature_dim, 1)

    def forward(self, x):
        flat = x.reshape(x.size(0), -1)
        return self.head(flat).squeeze(-1)


first_x, _ = next(iter(loader_for(train, shuffle=False)))
model = Model(first_x.reshape(first_x.size(0), -1).shape[1])
print(model, "| trainable:", sum(p.numel() for p in model.parameters() if p.requires_grad))

## 4 — One fold

Early stopping on validation AUC, `pos_weight` to compensate the class prior
(the training folds are ~56% deception, the holdouts ~47%).

In [ ]:
import copy
import torch
from lib.metrics import best_threshold, evaluate


def train_one_fold(fold, verbose=True):
    train_ids, val_ids, hold_ids = folds.subject(fold)
    dl_train, dl_val, dl_hold = (loader_for(train_ids, True),
                                 loader_for(val_ids, False),
                                 loader_for(hold_ids, False))

    net = Model(first_x.reshape(first_x.size(0), -1).shape[1])
    optimiser = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-2)
    loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=clips.class_weights(train_ids))

    best_score, best_state, best_thr, waited = -1.0, None, 0.5, 0
    for epoch in range(EPOCHS):
        net.train()
        for x, y in dl_train:
            optimiser.zero_grad()
            loss = loss_fn(net(x), y.float())
            loss.backward()
            optimiser.step()

        net.eval()
        probs, targets = [], []
        with torch.no_grad():
            for x, y in dl_val:
                probs += torch.sigmoid(net(x)).tolist()
                targets += y.tolist()
        score = evaluate(targets, probs)["auc"]
        score = score if score == score else 0.0          # NaN if one class only

        if score > best_score:
            best_score, best_state, waited = score, copy.deepcopy(net.state_dict()), 0
            best_thr = best_threshold(targets, probs)
        else:
            waited += 1
            if waited >= PATIENCE:
                break

    net.load_state_dict(best_state)
    net.eval()
    probs, targets, ids = [], [], []
    with torch.no_grad():
        for x, y in dl_hold:
            probs += torch.sigmoid(net(x)).tolist()
            targets += y.tolist()
    ids = clips.keep(hold_ids)

    metrics = evaluate(targets, probs, threshold=best_thr)
    if verbose:
        print(f"fold {fold}: ACC {metrics['acc']*100:5.1f}  F1 {metrics['f1']*100:5.1f}  "
              f"AUC {metrics['auc']*100:5.1f}  thr {best_thr:.2f}")
    return metrics, ids, targets, probs

## 5 — All five folds

This is the number you report: mean ± std over the folds. The pooled last line uses
every working-set clip exactly once and is the one to compare with the majority baseline.

In [ ]:
import numpy as np
import pandas as pd

results, pooled = [], []
for fold in range(1, FOLDS + 1):
    metrics, ids, targets, probs = train_one_fold(fold)
    results.append(metrics)
    pooled.append(pd.DataFrame({"clip_id": ids, "y_true": targets, "y_prob": probs}))

summary = pd.DataFrame(results)[["acc", "f1", "auc", "balanced_acc"]]
print(summary.round(3).to_string())
print(f"\nCV  ACC {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f}  "
      f"F1 {summary.f1.mean()*100:.1f}  AUC {summary.auc.mean()*100:.1f}")

In [ ]:
# pooled evaluation + the baseline that actually applies to these clips
from lib.metrics import evaluate, majority_baseline

preds = pd.concat(pooled, ignore_index=True)
overall = evaluate(preds.y_true, preds.y_prob)
base = majority_baseline(preds.y_true)
print(f"pooled ACC {overall['acc']*100:.1f}  balanced {overall['balanced_acc']*100:.1f}  "
      f"| majority baseline {base*100:.1f}")

# save for the paired comparison between directions (McNemar)
out = Path("preds"); out.mkdir(exist_ok=True)
preds.to_csv(out / f"cv_{DIRECTION}.csv", index=False)
print("saved", out / f"cv_{DIRECTION}.csv")

In [ ]:
# per-gender breakdown, aggregated over the fold holdouts
meta = pd.read_csv(DATA / "dataset.csv", dtype=str).set_index("clip_id")
preds["gender"] = preds.clip_id.map(meta.gender)
for gender, group in preds.groupby("gender"):
    m = evaluate(group.y_true, group.y_prob)
    print(f"{gender:7} n={m['n']:4d}  ACC {m['acc']*100:5.1f}  balanced {m['balanced_acc']*100:5.1f}")

## 6 — Row for the results board

The board is filled with the **cross-validation** numbers: mean and spread over the five folds,
plus the pooled result and the majority baseline that applies to these clips. Paste it in the section
`## Results board` of `lie-detector/README.md`, with your name and one line of interpretation.

In [ ]:
params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"| {DIRECTION} | <your name> | <model> | {params} "
      f"| {summary.acc.mean()*100:.1f} ± {summary.acc.std()*100:.1f} "
      f"| {summary.f1.mean()*100:.1f} | {summary.auc.mean()*100:.1f} "
      f"| {summary.balanced_acc.mean()*100:.1f} | {base*100:.1f} | <runtime per fold> |")